# Triagegeist challenge Solution Idea

In this Notebook i want to share my solution for the Tiragegeist challnge in Kaggle.
My proposal is the usage of a 3-stage pipeline to better flag ESI scores. 
In the first stage an XGBoost sees vital and immidiately assigns a acuity score (ESI). 
For all the patients that where flaged as safe by the XGBoost, we undergo a 2 Stage Pipeline
1. A small langauge model which will act as a semnatic router will read the major complaints and can quickly process all low-acuity cases in seconds 
2. The heavyweight auditor an example Llama-3 8B model provides the final, deep clinical reasoning strictly outputs a binary results "Yes,No" either upgrade or not the ESI of the patient to ESI2. 


### 1. Environment Setup

In [ ]:
import subprocess
import time

# 1. Install Ollama quietly (Skip if running locally)
!curl -fsSL https://ollama.com/install.sh | sh
print("Starting Ollama server...")
subprocess.Popen(["ollama", "serve"], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
time.sleep(5) 
!ollama pull llama3

# 2. Install required machine learning and HuggingFace libraries
!pip install -q ollama pandas xgboost scikit-learn tqdm transformers datasets accelerate
print("Environment Ready!")


from sklearn.metrics import cohen_kappa_score, f1_score


### 2. Data Preprocessing 

In [ ]:
import pandas as pd
import numpy as np
import xgboost as xgb
from sklearn.model_selection import train_test_split
from tqdm import tqdm
tqdm.pandas()

INPUT_DIR = '/kaggle/input/competitions/triagegeist'
OUTPUT_DIR = '/kaggle/working'

print("Loading data...")
train_df = pd.read_csv(f'{INPUT_DIR}/train.csv')
test_df = pd.read_csv(f'{INPUT_DIR}/test.csv')
complaints_df = pd.read_csv(f'{INPUT_DIR}/chief_complaints.csv')

# Merge chief complaints
train_df = train_df.merge(complaints_df, on='patient_id', how='left')
test_df = test_df.merge(complaints_df, on='patient_id', how='left')

# Drop Target Leakage
leakage_cols = ['disposition', 'ed_los_hours']
train_df = train_df.drop(columns=[c for c in leakage_cols if c in train_df.columns])

# Handle Clinical Missingness
vital_cols = ['systolic_bp', 'diastolic_bp', 'heart_rate', 'respiratory_rate', 'temperature_c', 'spo2']
for col in vital_cols:
    if col in train_df.columns:
        train_df[f'{col}_missing'] = train_df[col].isna().astype(int)
        test_df[f'{col}_missing'] = test_df[col].isna().astype(int)

# Categorical Encoding
object_cols = train_df.select_dtypes(include=['object', 'string']).columns.tolist()
object_cols = [col for col in object_cols if col not in ['patient_id', 'chief_complaint_raw']]

train_df = pd.get_dummies(train_df, columns=object_cols, drop_first=True)
test_df = pd.get_dummies(test_df, columns=object_cols, drop_first=True)

# Align columns
missing_cols = set(train_df.columns) - set(test_df.columns) - {'triage_acuity'}
for c in missing_cols:
    test_df[c] = 0

print(f"Train shape: {train_df.shape}, Test shape: {test_df.shape}")

### 3. Stage 1 - Fast Baseline (XGBoost)

In [ ]:
# XGBoost expects classes 0-4. ESI is 1-5, so subtract 1.
X_train = train_df.drop(columns=['patient_id', 'triage_acuity', 'chief_complaint_raw'])
y_train = train_df['triage_acuity'] - 1  
X_test = test_df[X_train.columns]

print("Training XGBoost Baseline...")
xgb_model = xgb.XGBClassifier(
    objective='multi:softprob',
    num_class=5,
    tree_method='hist',
    max_depth=6,
    learning_rate=0.05,
    n_estimators=300,
    random_state=42
)

xgb_model.fit(X_train, y_train)

# Generate baseline predictions (add 1 back to get ESI 1-5)
train_df['xgb_pred_acuity'] = xgb_model.predict(X_train) + 1
test_df['xgb_pred_acuity'] = xgb_model.predict(X_test) + 1
print("Stage 1: Tabular baseline complete.")

from sklearn.metrics import cohen_kappa_score, f1_score

# 1. Split training data to get a local validation score
X_train_split, X_val, y_train_split, y_val = train_test_split(
    X_train, y_train, test_size=0.2, random_state=42, stratify=y_train
)

# 2. Retrain just for metrics
xgb_eval = xgb.XGBClassifier(
    objective='multi:softprob', num_class=5, tree_method='hist',
    max_depth=6, learning_rate=0.05, n_estimators=300, random_state=42
)
xgb_eval.fit(X_train_split, y_train_split)
val_preds = xgb_eval.predict(X_val)

# 3. Calculate Metrics
qwk_score = cohen_kappa_score(y_val, val_preds, weights='quadratic')
f1_macro = f1_score(y_val, val_preds, average='macro')

print(f"Baseline Quadratic Weighted Kappa (QWK): {qwk_score:.3f}")
print(f"Baseline Macro F1-Score: {f1_macro:.3f}")

### 4. Stage 2-Fine-Tuning the Semantic Router (SML)

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification, Trainer, TrainingArguments
from datasets import Dataset

# 1. Create the Fine-Tuning Dataset
# Only train on cases that XGBoost thought were safe (ESI 3, 4, 5)
slm_train_df = train_df[train_df['xgb_pred_acuity'] >= 3].copy()
slm_train_df['chief_complaint_raw'] = slm_train_df['chief_complaint_raw'].fillna("none")

# Target: 1 if the true label was an emergency (ESI 1 or 2), 0 otherwise
slm_train_df['slm_label'] = (slm_train_df['triage_acuity'] <= 2).astype(int)

# Convert to HuggingFace Dataset
hf_dataset = Dataset.from_pandas(slm_train_df[['chief_complaint_raw', 'slm_label']])
hf_dataset = hf_dataset.rename_column("slm_label", "labels")
hf_dataset = hf_dataset.train_test_split(test_size=0.1, seed=42)

# 2. Tokenization (Using an ultra-fast small model)
model_id = "distilroberta-base" # You can swap this with any fast ~270M HuggingFace model
tokenizer = AutoTokenizer.from_pretrained(model_id)

def tokenize_function(examples):
    return tokenizer(examples["chief_complaint_raw"], padding="max_length", truncation=True, max_length=64)

tokenized_datasets = hf_dataset.map(tokenize_function, batched=True)

# 3. Initialize the Model
model = AutoModelForSequenceClassification.from_pretrained(model_id, num_labels=2)

# 4. Train the SLM
print("Fine-tuning the Semantic Router (SLM)...")
training_args = TrainingArguments(
    output_dir=f"{OUTPUT_DIR}/slm_router",
    eval_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=32,
    num_train_epochs=2, # Keep epochs low to prevent overfitting on Kaggle
    weight_decay=0.01,
    fp16=True, # Uses less VRAM on your RTX 3060
    report_to="none"
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_datasets["train"],
    eval_dataset=tokenized_datasets["test"],
)

trainer.train()
print("Stage 2: SLM Fine-tuning complete.")

### 5. Stage 3-The LLM Auditor Pipeline

In [ ]:
import ollama

# 1. Run the Semantic Router (SLM) on the Test Set
test_df['chief_complaint_raw'] = test_df['chief_complaint_raw'].fillna("none")
test_slm_df = test_df[test_df['xgb_pred_acuity'] >= 3].copy()

# Tokenize test cases and predict
print("Running Semantic Router on Test Set...")
test_dataset = Dataset.from_pandas(test_slm_df[['chief_complaint_raw']])
tokenized_test = test_dataset.map(tokenize_function, batched=True)

predictions = trainer.predict(tokenized_test)
# Convert logits to binary prediction (0 or 1)
test_slm_df['slm_flag'] = np.argmax(predictions.predictions, axis=-1)

# Cases flagged as '1' (Hidden Emergency) by the SLM
slm_flagged_indices = test_slm_df[test_slm_df['slm_flag'] == 1].index
cases_to_audit = test_df.loc[slm_flagged_indices].copy()

print(f"---> SLM Router dynamically flagged {len(cases_to_audit)} cases for LLM review. <---")

# 2. LLM Auditor Function (Llama-3)
def llm_safety_audit(complaint_text, current_acuity):
    prompt = f"""You are an expert emergency triage auditor. 
    The tabular model assigned this patient a low-acuity ESI score of {current_acuity}. 
    Chief complaint: "{complaint_text}"
    Does this text indicate a life-threatening emergency (e.g., stroke, heart attack, aortic dissection) that warrants upgrading to a high-acuity score (ESI 1 or 2)? 
    Answer strictly 'YES' or 'NO'."""
    
    try:
        response = ollama.chat(model='llama3', messages=[{'role': 'user', 'content': prompt}])
        return 'YES' in response['message']['content'].strip().upper()
    except Exception as e:
        return False 

# 3. Run the Auditor
if len(cases_to_audit) > 0:
    cases_to_audit['is_emergency'] = cases_to_audit.progress_apply(
        lambda row: llm_safety_audit(row['chief_complaint_raw'], row['xgb_pred_acuity']), axis=1)
    
    emergency_indices = cases_to_audit[cases_to_audit['is_emergency']].index
else:
    emergency_indices = []

# 4. Finalize Overrides
test_df['final_acuity_pred'] = test_df['xgb_pred_acuity']
if len(emergency_indices) > 0:
    test_df.loc[emergency_indices, 'final_acuity_pred'] = 2

print(f"Stage 3: LLM Auditor successfully upgraded {len(emergency_indices)} patients.")

### 6. Export& Bias Analysis

In [ ]:
# 1. Save Final Submission
submission = pd.DataFrame({
    'patient_id': test_df['patient_id'],
    'triage_acuity': test_df['final_acuity_pred'].astype(int)
})
submission.to_csv('submission.csv', index=False)
print("Saved submission.csv successfully!")

# 2. Bias Analysis Output
test_df['was_upgraded'] = test_df.index.isin(emergency_indices)
if 'sex_M' in test_df.columns:
    test_df['Gender'] = test_df['sex_M'].apply(lambda x: 'Male' if x == 1 else 'Female/Other')
    bias_table = pd.crosstab(test_df['Gender'], test_df['was_upgraded'])
    bias_table = bias_table.rename(columns={False: 'Not Upgraded', True: 'Upgraded by LLM'})
    
    if 'Upgraded by LLM' not in bias_table:
        bias_table['Upgraded by LLM'] = 0
        
    print("\n--- Bias Analysis: LLM Upgrades by Gender ---")
    print(bias_table)